# Test metering

To test metering you put records into your DynamoDB metering table.
The metering records must match dimensions from your product definition
and you provide a customer AWS Account Id from a customer that 
purchased your product.

In [ ]:
import boto3
import urllib.parse as urlparse 
import json
import shutil
import time

from datetime import datetime, timezone

## Settings
Replace `stack_name` and `customer_identifier` with the values from
your environment.

Values such as productId, DynamoDB table and Lambda hourly function will
be retrieved from your CloudFormation stack.

In [ ]:
# replace the value for stack_name with your value
stack_name = 'REPLACE_WITH_YOUR_STACKNAME'
# use the AWS account id for as customer_identifier
customer_identifier = 'REPLACE_WITH_YOUR_CUSTOMER_AWS_ACCOUNT_ID' # customers AWS Account id

In [ ]:
PROFILE = 'default' # aws profile to use
REGION = 'us-east-1'

SESSION = boto3.Session(profile_name=PROFILE, region_name=REGION)

## Functions

Some functions to get product definition, put metering records in DynamoDB and scan a DynamoDB table.

In [ ]:

def get_marketplace_product(product_id):
    client = SESSION.client('marketplace-catalog')

    response = client.describe_entity(
    Catalog='AWSMarketplace',
    EntityId=product_id
    )
    
    return response

def create_metering_item(customer_aws_account_id, dimension_name, dimension_value):
    #create_timestamp = f"{int(time.time())}" # Seconds precision, risk to overwrite records
    create_timestamp = f"{int(time.time_ns())}" # Nanosecond precision, virtually guarantees uniqueness
    item = {
        "create_timestamp": {
            "N": create_timestamp
        },
        "customerIdentifier": {
            "S": customer_aws_account_id
        },
        "dimension_usage": {
            "L": [
            {
                "M": {
                "dimension": {
                    "S": dimension_name
                },
                "value": {
                    "N": f'{dimension_value}'
                }
                }
            }
            ]
        },
        "metering_pending": {
            "S": "true"
        }
    }
    
    return item


def scan_table(table_name):
    dynamodb = boto3.resource('dynamodb')
    table = dynamodb.Table(table_name)
    
    response = table.scan()
    return response['Items']


## Get stack values

Get your product id, DynamoDB metering table names and Lambda hourly funtion name from your CloudFormation stack.

In [ ]:
product_id = None
metering_table_name = None
lambda_hourly_function_name = None

cfn = SESSION.client('cloudformation')

# get product_id from stack parameters
response = cfn.describe_stacks(StackName=stack_name)
parameters = response['Stacks'][0]['Parameters']
for parameter in parameters:
  if parameter['ParameterKey'] == 'ProductId':
    product_id = parameter['ParameterValue']
    break

# get other values from stack resources
paginator = cfn.get_paginator('list_stack_resources')

for page in paginator.paginate(StackName=stack_name):
    for resource in page['StackResourceSummaries']:
        #print(json.dumps(resource, indent=2, default=str))
        #print(f"{resource['ResourceType']}: {resource['LogicalResourceId']}")
        #if resource['LogicalResourceId'] == 'AWSMarketplaceSubscribers':
        #    print(f"Subscribers table name: {resource['PhysicalResourceId']}")
        if resource['LogicalResourceId'] == 'AWSMarketplaceMeteringRecords':
            #print(f"Metering table name: {resource['PhysicalResourceId']}")
            metering_table_name = resource['PhysicalResourceId']
        if resource['LogicalResourceId'] == 'Hourly':
            #print(f"Hourly Lambda function name: {resource['PhysicalResourceId']}")
            lambda_hourly_function_name = resource['PhysicalResourceId']


print(f"{'stack_name:':<35}{stack_name}")
print("-" * 60)
print(f"{'product_id:':<35}{product_id}")
print(f"{'metering_table_name:':<35}{metering_table_name}")
print(f"{'lambda_hourly_function_name:':<35}{lambda_hourly_function_name}")
print(f"{'customer_identifier:':<35}{customer_identifier}")

## Get product

Get your product description. In the description you find the usage dimension that you can use for metering.

In [ ]:
print(json.dumps(get_marketplace_product(product_id), indent=2, default=str))

## Create metering entries

Use `create_metering_item(CustomerAWSAccounId, Dimension)` to create
item and put them into the DynamoDB metering table.

**Update** `usage_dimensions` with your values. Get the dimensions from
your product definition you got earlier.

In [ ]:
# put your usage identifiers into the list which you want to meter
usage_dimensions = [
    {'usage_1': 1},
    {'usage_2': 2}
]

ddb = SESSION.client('dynamodb')
for dimension in usage_dimensions:
    for dimension_name, dimension_value in dimension.items():
        print(f"metering dimension: {dimension_name} value: {dimension_value}")
        item = create_metering_item(customer_identifier, dimension_name, dimension_value)
        print(f"metering item:\n{json.dumps(item, indent=2, default=str)}")

        response = ddb.put_item(
            TableName=metering_table_name,
            Item=item
        )
        print(json.dumps(response, indent=2, default=str))
        print('-' * 50)
        # time.sleep(2) # not required for nanoseconds precision

## Get entries from the metering table

Scan the metering table.

Unprocessed item look similar to:

```
{
  "dimension_usage": [
    {
      "dimension": "usage_2",
      "value": "3"
    }
  ],
  "metering_pending": "true",
  "create_timestamp": "1763396941",
  "customerIdentifier": "944681004585"
}
```

Processed records have a `metering_failed` boolean key and a `metering_response` key for example:

```
"metering_failed": false,
  "dimension_usage": [
    {
      "dimension": "usage_1",
      "value": "3"
    }
  ],
  "create_timestamp": "1763392067",
  "customerIdentifier": "944681004585",
  "metering_response": "{\"$metadata\":{\"httpStatusCode\":200,\"requestId\":\"828fd9e5-ee21-4b0f-9656-82a86b4e49c2\",\"attempts\":1,\"totalRetryDelay\":0},\"Results\":[{\"MeteringRecordId\":\"eb0e9db9-c90e-45fa-84c4-6239a68fb360\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_1\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"6756c73c-21fd-4821-a5d5-57a6d891f103\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_2\",\"Quantity\":6,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"c65d412f-aedf-4e89-afa3-a1d238277b63\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_3\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}}],\"UnprocessedRecords\":[]}"
}
```

In [ ]:
# Usage
items = scan_table(metering_table_name)
for item in items:
    print(json.dumps(item, indent=2, default=str))
    print('-' * 50)

print(f"\nscaned metering table: {metering_table_name}")

## Trigger metering hourly function

The Lambda function that meters hourly is automatically triggered by an
Amazon EventBridge rule. For testing purposes you can also
invoke the funtion manually.

In [ ]:
lmbd = SESSION.client('lambda')

response = lmbd.invoke(
    FunctionName=lambda_hourly_function_name,
    Payload=json.dumps({'start': 'metering'})
)

print(f"response:\n{json.dumps(response, indent=2, default=str)}")
print(f"Payload:\n{json.loads(response['Payload'].read())}")
